# Conditioning Audio Generation with AdaLN

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import numpy as np
import matplotlib.pyplot as plt

from stable_audio import get_pretrained_model
from stable_audio.models.conditioners import NumberConditioner, NumberEmbedder, ExpoFourierFeatures

# Device selection: MPS on Apple Silicon, CUDA if available, else CPU
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Using device: {device}")

# Load the pretrained Stable Audio 3 model
print("[*] Loading pretrained model: stabilityai/stable-audio-3-small-music ...")
model, model_config = get_pretrained_model("stabilityai/stable-audio-3-small-music")
model = model.eval().to(device)
print("[+] Model loaded successfully!")

# Inspect key architectural dimensions from model config
dit = model.model.model  # DiffusionTransformer
transformer = dit.transformer  # ContinuousTransformer
print(f"    - Transformer Dimension (dim):     {transformer.dim}")
print(f"    - Global Conditioning Dimension:   {dit.to_global_embed[0].in_features}")
print(f"    - Global Conditioning Type:        {dit.global_cond_type}")
print(f"    - Transformer Depth (Layers):      {len(transformer.layers)}")

No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention
No module named 'flash_attn'
flash_attn varlen/bert_padding not available, disabling varlen attention


/Users/mohammedali/Documents/stable-audio-3/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Using device: mps
[*] Loading pretrained model: stabilityai/stable-audio-3-small-music ...


/Users/mohammedali/Documents/stable-audio-3/.venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading weights: 100%|██████████| 134/134 [00:00<00:00, 5729.98it/s]


[+] Model loaded successfully!
    - Transformer Dimension (dim):     1024
    - Global Conditioning Dimension:   768
    - Global Conditioning Type:        adaLN
    - Transformer Depth (Layers):      20


## Phase 1:  Duration Conditioning Pipeline:

- Transform the single value of duration in seconds into embeddings that are in the conditioning space.
- The input dimention is 1, and output is 768. The output dimention selected to match the embeddings of tokens used for text propmts where in the cross attention conditioning it is added to the prompt embeddings.

![Duration Conditioning](./imgs/duration_conditioning.png)

- The resulted conditioning embeddings are passed to two paths for conditioning, one is the cross attention, and the other is the global conditioning which we will discuss in details.

#### Step 1: Scalar Duration Normalization:
Given target audio duration $d \in \mathbb{R}_{>0}$ (in seconds):
$$x_{\text{dur}} = \phi_{\text{norm}}(d) = \frac{\operatorname{clip}(d, \, t_{\min}, \, t_{\max}) - t_{\min}}{t_{\max} - t_{\min}} \in [0, 1]$$
where $t_{\min} = 0.0\text{ s}$ and $t_{\max} = 384.0\text{ s}$ (default max sequence length).

#### Step 2: Exponential Fourier Feature Embedding:
Deterministic multi-scale exponential frequencies with base frequency range $[f_{\min}, f_{\max}] = [0.5, 10000.0]$:
- Frequency dimension: $D = 256$, half-channels $H = D / 2 = 128$.
- For channel index $k \in \{0, 1, \dots, H - 1\}$:
$$\text{ramp}_k = \frac{k}{H - 1} \in [0, 1]$$
$$\omega_k = \exp\big(\text{ramp}_k \cdot (\ln f_{\max} - \ln f_{\min}) + \ln f_{\min}\big)$$
$$\theta_k(x_{\text{dur}}) = 2\pi \cdot x_{\text{dur}} \cdot \omega_k$$
$$\mathbf{e}_{\text{fourier}}(x_{\text{dur}}) = \Big[ \cos(\theta_0), \dots, \cos(\theta_{H-1}), \; \sin(\theta_0), \dots, \sin(\theta_{H-1}) \Big]^T \in \mathbb{R}^{256}$$

#### Step 3: Conditioner Output:
$$\mathbf{c}_{\text{dur}} = \mathbf{e}_{\text{fourier}}(x_{\text{dur}}) \mathbf{W}_{\text{dur}} + \mathbf{b}_{\text{dur}} \in \mathbb{R}^{768}$$


In [2]:
# Input target duration in seconds
duration_seconds = 15.0

# Extract the NumberConditioner module responsible for 'seconds_total'
duration_conditioner: NumberConditioner = model.conditioner.conditioners["seconds_total"]
embedder: NumberEmbedder = duration_conditioner.embedder
fourier_block: ExpoFourierFeatures = embedder.embedding[0]
linear_proj: nn.Linear = embedder.embedding[1]

# Perform normalization
d_clamped = max(duration_conditioner.min_val, min(duration_conditioner.max_val, float(duration_seconds)))
d_norm = (d_clamped - duration_conditioner.min_val) / (duration_conditioner.max_val - duration_conditioner.min_val)
d_norm_tensor = torch.tensor([d_norm], device=device, dtype=torch.float32)

num_freqs = fourier_block.dim // 2
# Compute Fourier features manually and with module
with torch.no_grad():
    fourier_features = fourier_block(d_norm_tensor)
    # Project to conditioner dimension (256 -> 768)
    c_dur = linear_proj(fourier_features)  # shape: [B, 1, 768]

print(f"Input duration:               {duration_seconds} seconds")
print(f"Normalized duration scalar:   {d_norm:.6f}  (in range [0, 1])")
print(f"Fourier features tensor shape: {tuple(fourier_features.shape)} (dim={fourier_features.shape[-1]})")
print(f"    Fourier min / max:             {fourier_features.min().item():.4f} / {fourier_features.max().item():.4f}")
print(f"    Fourier mean / std:            {fourier_features.mean().item():.4f} / {fourier_features.std().item():.4f}")
print(f"Duration embedding tensor:    shape={tuple(c_dur.shape)}")
print(f"    Embedding L2 norm:            {torch.linalg.vector_norm(c_dur).item():.4f}")
print(f"    Embedding mean / std:         {c_dur.mean().item():.4f} / {c_dur.std().item():.4f}")

Input duration:               15.0 seconds
Normalized duration scalar:   0.039062  (in range [0, 1])
Fourier features tensor shape: (1, 256) (dim=256)
    Fourier min / max:             -1.0000 / 1.0000
    Fourier mean / std:            0.1356 / 0.6953
Duration embedding tensor:    shape=(1, 768)
    Embedding L2 norm:            29.7705
    Embedding mean / std:         -0.0305 / 1.0745


---

## Phase 2: Produce Shared Embeddings used in every transformer head

- Project the Conditioning Embeddings into the Latent Space with dim=1024.
- Fusing the timestep embeddings in the latent space.
- Project into 6 different embeddings for the AdaLN-single parameters which are used accross the Transformer block $(c_{\gamma_s}, c_{\beta_s}, c_{\alpha_s}, c_{\gamma_f}, c_{\beta_f}, c_{\alpha_f})$

![Fusion and Shared Embeddings](./imgs/fusion_and_shared_embeddings.png)


#### Step 4: DiT Global Conditioning Projection (`to_global_embed`)
Inside the diffusion backbone (`DiffusionTransformer`):
$$\mathbf{g}_{\text{dur}} = \mathbf{W}_2 \cdot \operatorname{SiLU}(\mathbf{W}_1 \mathbf{c}_{\text{dur}}) \in \mathbb{R}^{1024}$$
where $\mathbf{W}_1 \in \mathbb{R}^{1024 \times 768}$ and $\mathbf{W}_2 \in \mathbb{R}^{1024 \times 1024}$ (linear layers without bias).

#### Step 5: Timestep Fusion
Continuous diffusion timestep $t \in [0, 1]$ is embedded via `to_timestep_embed` into $\mathbf{e}_t \in \mathbb{R}^{1024}$ using its `ExpoFourierFeatures` and a projection, then and summed:
$$\mathbf{g} = \mathbf{g}_{\text{dur}} + \mathbf{e}_t \in \mathbb{R}^{1024}$$

#### Step 6: AdaLN-Single Global Generator (`global_cond_embedder`)
In `ContinuousTransformer`, a single shared MLP projects $\mathbf{g}$ into $6 \times d_{\text{model}}$ dimensions:
$$\mathbf{u} = \mathbf{W}_4 \cdot \operatorname{SiLU}(\mathbf{W}_3 \mathbf{g} + \mathbf{b}_3) + \mathbf{b}_4 \in \mathbb{R}^{6144}$$


In [3]:
# Provide complete metadata for a conditioned generation sample
timestep_val = 0.5  # Midpoint diffusion timestep
t = torch.tensor([timestep_val], device=device, dtype=torch.float32)


with torch.no_grad():
    # 1. Project duration from 768 to 1024
    g_dur = dit.to_global_embed(c_dur)

    # 2. Compute timestep embedding in 1024
    t_cond = dit._t_to_logsnr_cond(t) if dit.timestep_features_logsnr else t
    t_cond = t_cond.to(c_dur.dtype)
    t_features = dit.timestep_features(t_cond[:, None])
    timestep_embed = dit.to_timestep_embed(t_features)

cos_sim = F.cosine_similarity(g_dur, timestep_embed, dim=-1).item()
# Fuse by element-wise addition as done in dit._forward()
fused_global = g_dur + timestep_embed

print(f"Timestep value: {timestep_val}")
print(f"Timestep Fourier features: {tuple(t_features.shape)}")
print("------Inside Latent Space------")
print(f"Timestep Embedding (e_t): {tuple(timestep_embed.shape)}")
print(f"Global Duration Embedding (g_dur): {tuple(g_dur.shape)}")
print(f"Cosine Similarity (g_dur · e_t):  {cos_sim:.4f}")
print(f"Fused Shape:                {tuple(fused_global.shape)}")
print(f"    Fused Norm:                 {torch.linalg.vector_norm(fused_global).item():.4f}")
print(f"    Fused Mean / Std:           {fused_global.mean().item():.4f} / {fused_global.std().item():.4f}")


Timestep value: 0.5
Timestep Fourier features: (1, 256)
------Inside Latent Space------
Timestep Embedding (e_t): (1, 1024)
Global Duration Embedding (g_dur): (1, 1024)
Cosine Similarity (g_dur · e_t):  -0.0954
Fused Shape:                (1, 1024)
    Fused Norm:                 11.7562
    Fused Mean / Std:           -0.0002 / 0.3676


In [4]:
# Produce the global modulation vector u used in every transformer Block
global_cond_embedder = transformer.global_cond_embedder

with torch.no_grad():
    global_mod = global_cond_embedder(fused_global)  # shape: [B, 6144]

print(f"Global modulation vector u:   shape={tuple(global_mod.shape)}")
print(f"    Modulation vector L2 norm:    {torch.linalg.vector_norm(global_mod).item():.4f}")
print(f"    Modulation vector mean / std: {global_mod.mean().item():.4f} / {global_mod.std().item():.4f}")

Global modulation vector u:   shape=(1, 6144)
    Modulation vector L2 norm:    31.4114
    Modulation vector mean / std: 0.0609 / 0.3961


---

## Phase 3: Inside a single Transformer Block (small_music has 20 Transformer Blocks)

- Inside each Transformer block in the DiT, the following things take place.
- Add the bias of that block to the shared global modulation eembeddings.
- Chunck into 6 embeddings with dim=1024, which is our required parameters: $(c_{\gamma_s}, c_{\beta_s}, c_{\alpha_s}, c_{\gamma_f}, c_{\beta_f}, c_{\alpha_f})$
- 3 of them will be used to modulate the self-attention process, and another 3 for modulating the feed-forward process.

![AdaLN-Single inside a Transformer Layer](./imgs/adaln_inside_transformer.png)



#### Step 7: Block-Specific Bias & 6-Way Chunking
For layer $l \in \{0, 1, \dots, L - 1\}$, the block contains a learnable bias parameter $\mathbf{w}_{\text{bias}}^{(l)} \in \mathbb{R}^{6144}$ (`to_scale_shift_gate`):
$$\mathbf{h}^{(l)} = \mathbf{u} + \mathbf{w}_{\text{bias}}^{(l)} \in \mathbb{R}^{6144}$$
$$\big[ \boldsymbol{\gamma}_{\text{self}}^{(l)}, \, \boldsymbol{\beta}_{\text{self}}^{(l)}, \, \boldsymbol{\alpha}_{\text{self}}^{(l)}, \, \boldsymbol{\gamma}_{\text{ff}}^{(l)}, \, \boldsymbol{\beta}_{\text{ff}}^{(l)}, \, \boldsymbol{\alpha}_{\text{ff}}^{(l)} \big] = \operatorname{chunk}_6\big(\mathbf{h}^{(l)}\big)$$
where each chunk is in $\mathbb{R}^{1 \times 1 \times 1024}$.

#### Step 8: Modulated Self-Attention Block
$$\hat{\mathbf{x}} = \operatorname{RMSNorm}(\mathbf{x})$$
$$\tilde{\mathbf{x}} = \hat{\mathbf{x}} \odot \big(1 + \boldsymbol{\gamma}_{\text{self}}^{(l)}\big) + \boldsymbol{\beta}_{\text{self}}^{(l)}$$
$$\mathbf{y}_{\text{attn}} = \operatorname{SelfAttention}\big(\tilde{\mathbf{x}}, \, \operatorname{RoPE}\big)$$
$$\mathbf{g}_{\text{self}} = \sigma\big(1 - \boldsymbol{\alpha}_{\text{self}}^{(l)}\big) = \frac{1}{1 + \exp\big(-(1 - \boldsymbol{\alpha}_{\text{self}}^{(l)})\big)}$$
$$\mathbf{x}_{\text{mid}} = \mathbf{x} + \operatorname{LayerScale}\big(\mathbf{y}_{\text{attn}} \odot \mathbf{g}_{\text{self}}\big)$$

#### Step 9: Modulated Feed-Forward Block
$$\hat{\mathbf{x}}_{\text{mid}} = \operatorname{RMSNorm}(\mathbf{x}_{\text{mid}})$$
$$\tilde{\mathbf{x}}_{\text{mid}} = \hat{\mathbf{x}}_{\text{mid}} \odot \big(1 + \boldsymbol{\gamma}_{\text{ff}}^{(l)}\big) + \boldsymbol{\beta}_{\text{ff}}^{(l)}$$
$$\mathbf{y}_{\text{ff}} = \operatorname{FeedForward}\big(\tilde{\mathbf{x}}_{\text{mid}}\big)$$
$$\mathbf{g}_{\text{ff}} = \sigma\big(1 - \boldsymbol{\alpha}_{\text{ff}}^{(l)}\big) = \frac{1}{1 + \exp\big(-(1 - \boldsymbol{\alpha}_{\text{ff}}^{(l)})\big)}$$
$$\mathbf{x}_{\text{out}} = \mathbf{x}_{\text{mid}} + \operatorname{LayerScale}\big(\mathbf{y}_{\text{ff}} \odot \mathbf{g}_{\text{ff}}\big)$$



In [5]:
# inside a Transformer block
layer_idx = 0
block = transformer.layers[layer_idx]
to_scale_shift_gate = block.to_scale_shift_gate  # Parameter of shape [6144]


with torch.no_grad():
    # Combine shared global modulation with layer-specific parameter
    h = (to_scale_shift_gate + global_mod).unsqueeze(1)  # shape: [B, 1, 6144]
    
    # 6-way decomposition
    scale_self, shift_self, gate_self, scale_ff, shift_ff, gate_ff = h.chunk(6, dim=-1)

# Compute effective residual gate multipliers: sigmoid(1 - gate)
effective_gate_self = torch.sigmoid(1 - gate_self)
effective_gate_ff = torch.sigmoid(1 - gate_ff)

print(f" Global modulation u norm:     {torch.linalg.vector_norm(global_mod).item():.4f}")
print(f"6-WAY CHUNKED ADALN PARAMETERS FOR LAYER {layer_idx}:")
print("-" * 80)
print(f"  1. scale_self (γ_attn): shape={tuple(scale_self.shape)} | mean={scale_self.mean().item():+.4f} | std={scale_self.std().item():.4f}")
print(f"  2. shift_self (β_attn): shape={tuple(shift_self.shape)} | mean={shift_self.mean().item():+.4f} | std={shift_self.std().item():.4f}")
print(f"  3. gate_self  (α_attn): shape={tuple(gate_self.shape)}  | mean={gate_self.mean().item():+.4f}  | std={gate_self.std().item():.4f}")
print(f"     └─► Effective Residual Gate σ(1 - α_attn):       mean={effective_gate_self.mean().item():.4f} (range [{effective_gate_self.min().item():.4f}, {effective_gate_self.max().item():.4f}])")
print(f"  4. scale_ff   (γ_ff):   shape={tuple(scale_ff.shape)}   | mean={scale_ff.mean().item():+.4f}   | std={scale_ff.std().item():.4f}")
print(f"  5. shift_ff   (β_ff):   shape={tuple(shift_ff.shape)}   | mean={shift_ff.mean().item():+.4f}   | std={shift_ff.std().item():.4f}")
print(f"  6. gate_ff    (α_ff):   shape={tuple(gate_ff.shape)}    | mean={gate_ff.mean().item():+.4f}    | std={gate_ff.std().item():.4f}")
print(f"     └─► Effective Residual Gate σ(1 - α_ff):         mean={effective_gate_ff.mean().item():.4f} (range [{effective_gate_ff.min().item():.4f}, {effective_gate_ff.max().item():.4f}])")

 Global modulation u norm:     31.4114
6-WAY CHUNKED ADALN PARAMETERS FOR LAYER 0:
--------------------------------------------------------------------------------
  1. scale_self (γ_attn): shape=(1, 1, 1024) | mean=+0.2783 | std=0.2660
  2. shift_self (β_attn): shape=(1, 1, 1024) | mean=+0.0013 | std=0.3616
  3. gate_self  (α_attn): shape=(1, 1, 1024)  | mean=+0.0995  | std=0.4969
     └─► Effective Residual Gate σ(1 - α_attn):       mean=0.7023 (range [0.3290, 0.9907])
  4. scale_ff   (γ_ff):   shape=(1, 1, 1024)   | mean=-0.3122   | std=0.1923
  5. shift_ff   (β_ff):   shape=(1, 1, 1024)   | mean=+0.0068   | std=0.2460
  6. gate_ff    (α_ff):   shape=(1, 1, 1024)    | mean=+0.2646    | std=0.3956
     └─► Effective Residual Gate σ(1 - α_ff):         mean=0.6709 (range [0.3557, 0.9925])


In [6]:
# ==============================================================================
# PHASE 7: ADALN MODULATION IN ACTION INSIDE TRANSFORMER BLOCK
# ==============================================================================

# Create a realistic token sequence activation tensor x
seq_len = 256  # 256 latent frames
batch_size = 1
x = torch.randn(batch_size, seq_len, transformer.dim, device=device)

print(f"Input activation x:           shape={tuple(x.shape)} | mean={x.mean().item():.4f} | std={x.std().item():.4f}")

with torch.no_grad():
    # 1. Pre-norm using RMSNorm
    residual = x
    x_norm = block.pre_norm(x)
    print(f"After RMSNorm (pre_norm):     shape={tuple(x_norm.shape)} | mean={x_norm.mean().item():.4f} | std={x_norm.std().item():.4f}")
    
    # 2. AdaLN Modulation: x_norm * (1 + scale_self) + shift_self
    x_modulated = x_norm * (1 + scale_self) + shift_self
    print(f"After AdaLN Modulation:       shape={tuple(x_modulated.shape)} | mean={x_modulated.mean().item():.4f} | std={x_modulated.std().item():.4f}")
    print(f"    └─► Mean shifted from {x_norm.mean().item():.4f} to {x_modulated.mean().item():.4f}")
    print(f"    └─► Std scaled from   {x_norm.std().item():.4f} to {x_modulated.std().item():.4f}")

with torch.no_grad():
    # Self-attention forward pass
    attn_out = block.self_attn(x_modulated)
    print(f"Raw Attention Output:         shape={tuple(attn_out.shape)} | norm={torch.linalg.vector_norm(attn_out).item():.4f}")
    
    # Apply gated residual: attn_out * sigmoid(1 - gate_self)
    gated_attn_out = attn_out * effective_gate_self
    scaled_attn_out = block.self_attn_scale(gated_attn_out)
    x_mid = residual + scaled_attn_out
    print(f"Gated Attention Output:       norm={torch.linalg.vector_norm(gated_attn_out).item():.4f}")
    print(f"After Attention Residual Add: norm={torch.linalg.vector_norm(x_mid).item():.4f}")

with torch.no_grad():
    residual_ff = x_mid
    x_ff_norm = block.ff_norm(x_mid)    
    # AdaLN Modulation: x_ff_norm * (1 + scale_ff) + shift_ff
    x_ff_modulated = x_ff_norm * (1 + scale_ff) + shift_ff
    print(f"After AdaLN Modulation:       shape={tuple(x_ff_modulated.shape)} | mean={x_ff_modulated.mean().item():.4f} | std={x_ff_modulated.std().item():.4f}")


with torch.no_grad():
    ff_out = block.ff(x_ff_modulated)
    gated_ff_out = ff_out * effective_gate_ff
    scaled_ff_out = block.ff_scale(gated_ff_out)
    x_out = residual_ff + scaled_ff_out
    print(f"Raw FFN Output:               shape={tuple(ff_out.shape)} | norm={torch.linalg.vector_norm(ff_out).item():.4f}")
    print(f"Gated FFN Output:             norm={torch.linalg.vector_norm(gated_ff_out).item():.4f}")
    print(f"Final Block Output:           shape={tuple(x_out.shape)} | norm={torch.linalg.vector_norm(x_out).item():.4f}")

    # Mathematical equivalence verification with block.forward()
    direct_block_out = block(x, global_cond=global_mod)
    max_diff = (x_out - direct_block_out).abs().max().item()
    print(f"\n  [Mathematical Equivalence Check]:")
    print(f"    max_abs_diff(manual_step_by_step, block.forward) = {max_diff:.8e}")
    assert max_diff < 1e-5, f"Discrepancy detected: {max_diff}"
    print("    [+] Step-by-step walkthrough is 100% mathematically identical to the actual block!")


Input activation x:           shape=(1, 256, 1024) | mean=-0.0007 | std=1.0011
After RMSNorm (pre_norm):     shape=(1, 256, 1024) | mean=-0.0004 | std=0.5149
After AdaLN Modulation:       shape=(1, 256, 1024) | mean=0.0008 | std=0.7536
    └─► Mean shifted from -0.0004 to 0.0008
    └─► Std scaled from   0.5149 to 0.7536
Raw Attention Output:         shape=(1, 256, 1024) | norm=4419.1431
Gated Attention Output:       norm=3555.7053
After Attention Residual Add: norm=3587.5137
After AdaLN Modulation:       shape=(1, 256, 1024) | mean=0.0016 | std=0.2097
Raw FFN Output:               shape=(1, 256, 1024) | norm=3887.2327
Gated FFN Output:             norm=2772.9146
Final Block Output:           shape=(1, 256, 1024) | norm=5542.8750

  [Mathematical Equivalence Check]:
    max_abs_diff(manual_step_by_step, block.forward) = 0.00000000e+00
    [+] Step-by-step walkthrough is 100% mathematically identical to the actual block!
